In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.cluster import DBSCAN, HDBSCAN
from sklearn.preprocessing import StandardScaler

from src.analysis.dim_reducer import reduce_dimensionality
from src.util.datasets import load_mnist_dataset

DATASET_PATH = Path("datasets/wine_quality/wine+quality/winequality-red.csv")
df = pd.read_csv(DATASET_PATH, sep=";")
df = df.reset_index(drop=True)
df["row_id"] = df.index

In [ ]:
X = df.drop(columns=["quality", "row_id"]).values
scaler = StandardScaler()
X_scaled: pd.DataFrame = scaler.fit_transform(X)

In [ ]:
mnist_data = load_mnist_dataset()
# mnist_x_combined = np.concatenate([mnist_data[0], mnist_data[2]], axis=0)
mnist_x_combined = mnist_data[0]  # only use training data for now
mnist_df = pd.DataFrame(mnist_x_combined, columns=[f"pixel_{i}" for i in range(784)])
# mnist_df["label"] = mnist_data[1] + mnist_data[3]
mnist_df["label"] = mnist_data[1]
mnist_df["row_id"] = mnist_df.index
mnist_x_scaled = scaler.fit_transform(mnist_x_combined)
mnist_X = mnist_df.drop(columns=["label", "row_id"]).values


In [ ]:
import umap
reducer = umap.UMAP(n_components=10, n_neighbors=30, min_dist=0.0)
X_umap = reducer.fit_transform(X_scaled)
model = HDBSCAN(min_cluster_size=15, min_samples=5)
labels = model.fit_predict(X_umap)


df["cluster"] = labels

df

In [ ]:
X_used = X_scaled
# X_used = reduce_dimensionality(method="PCA", X=X_used, n_components=10)

# min_samples_fraction = 0.005
# min_samples = max(2, int(min_samples_fraction * len(X_used)))

min_samples = 5

fraction_outliers = 0.3
min_clusters = 3

eps_start = 0.5
eps_factor = 0.5
counts = []

for i in range(1, 50):
    eps = eps_start + i * eps_factor

    model = DBSCAN(eps=eps, min_samples=min_samples)
    labels = model.fit_predict(X_used)

    value_counts = pd.Series(labels).value_counts().sort_index()

    outliers = value_counts.get(-1, 0)
    outlier_fraction = outliers / len(X_used)

    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)

    print(
        f"eps={eps:.2f}, clusters={n_clusters}, "
        f"outliers={outlier_fraction:.2%}, counts={dict(value_counts)}"
    )

    # if outlier_fraction <= fraction_outliers and n_clusters >= min_clusters:
    counts.append((eps, value_counts))

print("accepted:", counts)

In [ ]:
eps_cluster_counts = [(eps, len(value_counts)) for eps, value_counts in counts]
eps_cluster_counts

max_clusters_eps = max(eps_cluster_counts, key=lambda x: x[1])

In [ ]:
eps = max_clusters_eps[0]
# eps = 1.5
for i in range (1, 20):
    min_samples = i * 2
    
    model = DBSCAN(eps=eps, min_samples=min_samples)
    labels = model.fit_predict(X_used)

    value_counts = pd.Series(labels).value_counts().sort_index()

    outliers = value_counts.get(-1, 0)
    outlier_fraction = outliers / len(X_used)

    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)

    print(
        f"min_samples={min_samples}, clusters={n_clusters}, "
        f"outliers={outlier_fraction:.2%}, counts={dict(value_counts)}"
    )

    if outlier_fraction <= fraction_outliers and n_clusters >= min_clusters:
        counts.append((eps, value_counts))

print("accepted:", counts)